# 🎙️ BioEcho v2 — Notebook 1: Voice Biomarker Model
**Detects:** Parkinson's · Depression · Respiratory illness · Cognitive load

| Feature | Detail |
|---|---|
| Wav2Vec2 | Large (1024-dim) BF16 |
| LoRA | r=16 on all attention layers |
| Optimizer | AdamW-8bit (bitsandbytes) |
| Precision | BF16 (single GPU, no DataParallel) |
| Augmentation | Temporal jitter + CutMix |
| Loss | Gaussian NLL + Kendall-Gal task weighting |
| Calibration | ECE plots + reliability diagrams |
| EMA | decay=0.9995 |
| Targets | 8 tasks |
| Export | FP32→INT8 ONNX |

**Datasets:** RAVDESS · CREMA-D · MDVR-KCL (Kaggle input) · UCI synthesised
**Skipped:** mPower (requires DUA) · PC-GITA (requires DUA)


In [ ]:
# ── Kaggle API credentials
import os, json, subprocess
from pathlib import Path

KAGGLE_USERNAME = os.environ.get('KAGGLE_USERNAME', 'saroopmandal')
KAGGLE_KEY = os.environ.get('KAGGLE_KEY', '')

kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)
creds_path = kaggle_dir / 'kaggle.json'
with open(creds_path, 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
creds_path.chmod(0o600)
print(f'✅ Kaggle credentials set: {KAGGLE_USERNAME}')


In [ ]:
import subprocess

def run(cmd, timeout=900):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    if r.returncode != 0 and r.stderr:
        print(f'WARN [{cmd[:50]}]: {r.stderr[:200]}')
    return r

run('pip install -q --upgrade pip setuptools wheel')
run('pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu121', timeout=600)
run('pip install -q bitsandbytes>=0.43.0')
run('pip install -q peft>=0.10.0')
run('pip install -q transformers>=4.40.0 accelerate>=0.29.0')
run('pip install -q opensmile librosa soundfile noisereduce audiomentations')
run('pip install -q scikit-learn scipy einops rich matplotlib seaborn pandas torchmetrics')
run('pip install -q onnx onnxruntime-gpu')

# Flash Attention 2 — optional speedup
r_fa = run('pip install -q flash-attn --no-build-isolation', timeout=600)
FA2_OK = r_fa.returncode == 0
print(f'Flash-Attn 2: {"✅" if FA2_OK else "⚠️ fallback to SDPA"}')
print('\n✅ All packages installed')


In [ ]:
import os, gc, json, time, math, warnings, random, shutil
from pathlib import Path
from copy import deepcopy
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from rich.console import Console
from rich.table import Table

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
# GradScaler: use torch.amp.GradScaler (torch.cuda.amp is deprecated)

import librosa
import soundfile as sf
import noisereduce as nr
import opensmile
from audiomentations import Compose, AddGaussianNoise, TimeStretch, PitchShift, Shift

from transformers import Wav2Vec2Model, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

import bitsandbytes as bnb
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import calibration_curve
import onnx
from onnxruntime.quantization import quantize_dynamic, QuantType
import onnxruntime as ort

warnings.filterwarnings('ignore')
console = Console()

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

DEVICE   = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
DTYPE    = torch.float16
FA2_OK   = False

NUM_GPUS = torch.cuda.device_count()
if torch.cuda.is_available():
    for i in range(NUM_GPUS):
        props = torch.cuda.get_device_properties(i)
        console.print(f'[green]GPU {i}: {props.name} | {props.total_memory/1e9:.1f} GB[/]')
console.print(f'[bold green]Using: {DEVICE} | dtype: {DTYPE} | GPUs: {NUM_GPUS}[/]')


In [ ]:
@dataclass
class VoiceConfig:
    # ── Paths
    base_dir:    str = '/kaggle/working/bioecho/voice'
    data_dir:    str = '/kaggle/working/bioecho/voice/data'
    ckpt_dir:    str = '/kaggle/working/bioecho/voice/checkpoints'
    out_dir:     str = '/kaggle/working/bioecho/voice/outputs'
    resume_ckpt: str = ''

    # ── Audio
    sample_rate:  int   = 16000
    max_duration: float = 8.0        # was 10.0 — saves 20% compute per sample
    smile_dim:    int   = 6373

    # ── Wav2Vec2-Large
    w2v_model:         str = 'facebook/wav2vec2-large'
    w2v_dim:           int = 1024
    w2v_freeze_layers: int = 20      # was 16 — freeze more = faster forward pass

    # ── LoRA — smaller for prototype speed
    lora_r:      int   = 8           # was 16 — half params, fine for prototype
    lora_alpha:  int   = 16          # keep ratio lora_alpha/lora_r = 2
    lora_dropout: float = 0.05
    lora_target_modules: List[str] = field(default_factory=lambda: [
        'q_proj', 'v_proj'          # was q/k/v/out — fewer targets = faster
    ])

    # ── Model dims
    embed_dim:  int   = 256
    hidden_dim: int   = 512
    dropout:    float = 0.3

    # ── Tasks
    binary_tasks: List[str] = field(default_factory=lambda: [
        'parkinsons', 'depression', 'respiratory'
    ])
    regress_tasks: List[str] = field(default_factory=lambda: [
        'emotion_valence', 'emotion_arousal',
        'bp_systolic', 'hrv_sdnn', 'cognitive_load',
    ])

    # ── Training — tuned for dual T4
    # DataParallel doubles effective batch, so LRs are halved vs single-GPU
    batch_size:   int   = 32         # was 8 — larger batch uses both GPUs better
    grad_accum:   int   = 1          # was 4 — reduced since batch already doubled
    epochs:       int   = 10         # was 40 — prototype
    lr:           float = 5e-5       # was 1e-4 — HALVED for dual GPU NaN fix
    w2v_lr:       float = 2e-6       # was 5e-6 — HALVED for dual GPU NaN fix
    weight_decay: float = 1e-4
    warmup_steps: int   = 200
    grad_clip:    float = 0.5        # was 1.0 — tighter clip prevents NaN
    patience:     int   = 7          # was 10
    ema_decay:    float = 0.9995

C = VoiceConfig()
C.all_tasks = C.binary_tasks + C.regress_tasks

for d in [C.data_dir, C.ckpt_dir, C.out_dir]:
    Path(d).mkdir(parents=True, exist_ok=True)

console.print(f'[cyan]Wav2Vec2 Large | LoRA r={C.lora_r} | BF16 | EMA={C.ema_decay}[/]')
console.print(f'[cyan]Tasks ({len(C.all_tasks)}): {C.all_tasks}[/]')
console.print(f'[cyan]LR={C.lr} | w2v_lr={C.w2v_lr} | batch={C.batch_size} | epochs={C.epochs}[/]')


In [ ]:
# ── Checkpoint utilities
CKPT_DIR = Path(C.ckpt_dir)

def save_checkpoint(ckpt_dir, epoch, model_state, ema_shadow,
                    opt_state, sch_state, scaler_state, history, val_loss):
    """Save full training state for auto-resume."""
    ckpt_dir = Path(ckpt_dir)
    p = ckpt_dir / f'voice_epoch_{epoch:03d}.pt'
    torch.save({
        'epoch': epoch, 'model_state': model_state,
        'ema_shadow': ema_shadow, 'opt_state': opt_state,
        'sch_state': sch_state, 'scaler_state': scaler_state,
        'history': history, 'best_val': val_loss
    }, p)
    # Keep only last 2 checkpoints
    old = sorted(ckpt_dir.glob('voice_epoch_*.pt'),
                 key=lambda x: int(x.stem.split('_')[-1]))[:-2]
    for o in old:
        o.unlink(missing_ok=True)
    return p

def load_checkpoint(ckpt_path, model, optimizer, scheduler, scaler):
    """Load training state from checkpoint."""
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['opt_state'])
    scheduler.load_state_dict(ckpt['sch_state'])
    scaler.load_state_dict(ckpt['scaler_state'])
    return ckpt['epoch'], 0, ckpt['ema_shadow'], ckpt['history'], ckpt['best_val']

# Check for existing checkpoint to resume
existing_ckpts = sorted(CKPT_DIR.glob('voice_epoch_*.pt'),
                        key=lambda x: int(x.stem.split('_')[-1]))
if existing_ckpts:
    C.resume_ckpt = str(existing_ckpts[-1])
    console.print(f'[yellow]▶ Resume from: {existing_ckpts[-1].name}[/]')
else:
    console.print('[green]✅ Fresh training run[/]')


In [ ]:
import urllib.request, zipfile, tarfile, csv

DATA = Path(C.data_dir)

def download(url, dest, desc=''):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 1000:
        console.print(f'[yellow]Skip (exists): {dest.name}[/]'); return True
    dest.parent.mkdir(parents=True, exist_ok=True)
    try:
        console.print(f'[cyan]Downloading {desc or dest.name}...[/]')
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=120) as r, open(dest, 'wb') as f:
            total = int(r.headers.get('Content-Length', 0))
            downloaded = 0
            while chunk := r.read(1024 * 1024):
                f.write(chunk); downloaded += len(chunk)
                pct = f'{100*downloaded/total:.1f}%' if total else f'{downloaded/1e6:.1f} MB'
                print(f'\r  {pct}', end='', flush=True)
        print()
        console.print(f'[green]✅ {dest.name} ({dest.stat().st_size/1e6:.1f} MB)[/]')
        return True
    except Exception as e:
        print(); console.print(f'[red]❌ {desc}: {e}[/]'); return False

def extract(archive, dest, overwrite=False):
    dest = Path(dest)
    if dest.exists() and not overwrite:
        console.print(f'[yellow]Skip extract (exists): {dest}[/]'); return
    dest.mkdir(parents=True, exist_ok=True)
    try:
        if Path(archive).suffix == '.zip':
            with zipfile.ZipFile(archive) as z: z.extractall(dest)
        elif Path(archive).suffix in ['.gz', '.tgz']:
            with tarfile.open(archive) as t: t.extractall(dest)
        console.print(f'[green]✅ Extracted → {dest}[/]')
    except Exception as e:
        console.print(f'[red]Extract error: {e}[/]')

# ── 1. RAVDESS (Kaggle input — skip download)
USE_RAVDESS = False
for _rav_path in [Path('/kaggle/input/ravdess-emotional-speech-audio'),
                  Path('/kaggle/input/ravdess-emotional-speech-audio/audio_speech_actors_01-24'),
                  Path('/kaggle/input/ravdess'),
                  Path('/kaggle/input/ravdess-emotional-song-audio')]:
    if _rav_path.exists():
        rav_dir = DATA / 'ravdess'
        if not rav_dir.exists(): rav_dir.symlink_to(_rav_path)
        USE_RAVDESS = True
        console.print(f'[green]\u2705 RAVDESS found: {_rav_path}[/]')
        break
if not USE_RAVDESS:
    console.print('[yellow]RAVDESS not found in /kaggle/input/ — add it as a dataset[/]')
# ── 2. CREMA-D — from Kaggle input (GitHub version gives corrupt files)
# Add dataset: https://www.kaggle.com/datasets/ejlok1/cremad
cremad_dir = None
for candidate in [
    Path('/kaggle/input/cremad'),
    Path('/kaggle/input/crowd-sourced-emotional-multimodal-actors-dataset-crema-d'),
    Path('/kaggle/input/datasets/ejlok1/cremad'),
]:
    if candidate.exists() and len(list(candidate.rglob('*.wav'))) >= 10:
        cremad_dir = candidate
        break

USE_CREMAD = cremad_dir is not None
if USE_CREMAD:
    n = len(list(cremad_dir.rglob('*.wav')))
    console.print(f'[green]✅ CREMA-D found: {n} wav files at {cremad_dir}[/]')
else:
    console.print('[yellow]⚠️ CREMA-D not found — add dataset ejlok1/cremad in Kaggle input[/]')

# ── 3. MDVR-KCL — already in Kaggle input
mdvrkcl_dir = Path('/kaggle/input/datasets/nutansingh/mdvr-kcl-dataset')
USE_MDVRKCL = mdvrkcl_dir.exists() and len(list(mdvrkcl_dir.rglob('*.wav'))) >= 10
if USE_MDVRKCL:
    n = len(list(mdvrkcl_dir.rglob('*.wav')))
    console.print(f'[green]✅ MDVR-KCL found: {n} wav files[/]')
else:
    console.print('[yellow]⚠️ MDVR-KCL not found[/]')

# ── 4. UCI Parkinson's Voice Dataset
uci_dir  = DATA / 'uci_parkinsons'
uci_data = uci_dir / 'parkinsons.data'
USE_UCI  = download(
    'https://archive.ics.uci.edu/ml/machine-learning-databases/parkinsons/parkinsons.data',
    uci_data, 'UCI Parkinson Voice'
)
if USE_UCI:
    uci_wav_dir = uci_dir / 'wav'; uci_wav_dir.mkdir(exist_ok=True)
    existing = list(uci_wav_dir.glob('*.wav'))
    if len(existing) >= 190:
        console.print(f'[yellow]Skip UCI synthesis (exists): {len(existing)} wav files[/]')
    else:
        console.print('[cyan]Synthesising UCI wav files from dysphonia features...[/]')
        sr_uci, dur = 22050, 3.0
        t_uci = np.linspace(0, dur, int(sr_uci * dur))
        count = 0
        with open(uci_data) as f:
            for i, row in enumerate(csv.DictReader(f)):
                is_pd   = int(row['status']) == 1
                fo      = float(row['MDVP:Fo(Hz)'])
                jitter  = float(row['MDVP:Jitter(%)']) / 100.0
                shimmer = float(row['MDVP:Shimmer'])
                hnr     = float(row['HNR'])
                sig = np.zeros_like(t_uci)
                for h in range(1, 7):
                    freq_noise = 1 + jitter * np.random.randn(len(t_uci))
                    phase = 2 * np.pi * h * fo * np.cumsum(freq_noise) / sr_uci
                    amp   = 1 + shimmer * np.random.randn(len(t_uci))
                    sig  += (amp / h) * np.sin(phase)
                sig += 10 ** (-hnr / 20) * np.random.randn(len(t_uci))
                if is_pd:
                    sig *= 1 + 0.15 * np.sin(2 * np.pi * np.random.uniform(4, 6) * t_uci)
                sig = sig / (np.abs(sig).max() + 1e-8) * 0.85
                label = 'pd' if is_pd else 'hc'
                sf.write(uci_wav_dir / f'{label}_{i:04d}.wav', sig.astype(np.float32), sr_uci)
                count += 1
        console.print(f'[green]✅ UCI: synthesised {count} wav files[/]')

# ── 5. mPower — skipped (requires Synapse DUA approval)
USE_MPOWER = False
console.print('[yellow]⏭  mPower skipped (requires DUA)[/]')

# ── 6. PC-GITA — skipped (requires signed DUA from author)
USE_PCGITA = False
console.print('[yellow]⏭  PC-GITA skipped (requires DUA)[/]')

console.print('\n[bold]Dataset status:[/]')
console.print(f'  RAVDESS  : {"[green]✅[/]" if USE_RAVDESS else "[red]❌[/]"}')
console.print(f'  CREMA-D  : {"[green]✅[/]" if USE_CREMAD  else "[yellow]⚠️ add ejlok1/cremad[/]"}')
console.print(f'  MDVR-KCL : {"[green]✅[/]" if USE_MDVRKCL else "[yellow]⚠️ not found[/]"}')
console.print(f'  UCI      : {"[green]✅[/]" if USE_UCI     else "[red]❌[/]"}')
console.print('[bold green]\n✅ Dataset downloads complete[/]')

In [ ]:
records = []

RAVDESS_VALENCE  = {1:0.0,2:0.2,3:0.9,4:-0.8,5:-0.5,6:-0.7,7:-0.6,8:0.6}
RAVDESS_AROUSAL  = {1:0.0,2:0.1,3:0.7,4:0.3,5:0.9,6:0.8,7:0.5,8:0.8}
DEPRESSION_EMO   = {4, 6, 7}

if USE_RAVDESS:
    for wav in (DATA/'ravdess').rglob('*.wav'):
        parts = wav.stem.split('-')
        if len(parts) < 7: continue
        try: emo = int(parts[2])
        except ValueError: continue
        records.append({
            'path': str(wav), 'parkinsons': 0,
            'depression': int(emo in DEPRESSION_EMO), 'respiratory': 0,
            'emotion_valence': RAVDESS_VALENCE.get(emo, 0.0),
            'emotion_arousal': RAVDESS_AROUSAL.get(emo, 0.0),
            'bp_systolic': 120.0 + np.random.randn() * 10,
            'hrv_sdnn': 45.0 + np.random.randn() * 15,
            'cognitive_load': np.random.uniform(0, 1),
            'source': 'ravdess'
        })
    console.print(f'[green]RAVDESS: {sum(1 for r in records if r["source"]=="ravdess")} records[/]')

CREMAD_VALENCE   = {'ANG':-0.5,'DIS':-0.6,'FEA':-0.7,'HAP':0.9,'NEU':0.0,'SAD':-0.8}
CREMAD_AROUSAL   = {'ANG':0.9,'DIS':0.5,'FEA':0.8,'HAP':0.7,'NEU':0.0,'SAD':0.3}
CREMAD_DEPRESSED = {'SAD','DIS','FEA'}

if USE_CREMAD:
    cremad_ok = 0
    for wav in cremad_dir.rglob('*.wav'):
        try:
            sf.info(str(wav))
        except Exception:
            continue
        parts = wav.stem.split('_')
        if len(parts) < 3: continue
        emo = parts[2]
        records.append({
            'path': str(wav), 'parkinsons': 0,
            'depression': int(emo in CREMAD_DEPRESSED), 'respiratory': 0,
            'emotion_valence': CREMAD_VALENCE.get(emo, 0.0),
            'emotion_arousal': CREMAD_AROUSAL.get(emo, 0.0),
            'bp_systolic': 120.0 + np.random.randn() * 10,
            'hrv_sdnn': 45.0 + np.random.randn() * 15,
            'cognitive_load': np.random.uniform(0, 1),
            'source': 'cremad'
        })
        cremad_ok += 1
    console.print(f'[green]CREMA-D: {cremad_ok} valid records[/]')

if USE_MDVRKCL:
    mdvrkcl_ok = 0
    for wav in mdvrkcl_dir.rglob('*.wav'):
        is_pd = '/PD/' in str(wav)
        records.append({
            'path': str(wav), 'parkinsons': int(is_pd),
            'depression': 0, 'respiratory': 0,
            'emotion_valence': 0.0, 'emotion_arousal': 0.0,
            'bp_systolic': 130.0 if is_pd else 118.0,
            'hrv_sdnn': 30.0 if is_pd else 50.0,
            'cognitive_load': 0.55 if is_pd else 0.25,
            'source': 'mdvrkcl'
        })
        mdvrkcl_ok += 1
    console.print(f'[green]MDVR-KCL: {mdvrkcl_ok} records[/]')

if USE_UCI:
    uci_ok = 0
    for wav in (uci_dir / 'wav').glob('*.wav'):
        is_pd = wav.stem.startswith('pd_')
        records.append({
            'path': str(wav), 'parkinsons': int(is_pd),
            'depression': 0, 'respiratory': 0,
            'emotion_valence': 0.0, 'emotion_arousal': 0.0,
            'bp_systolic': 132.0 if is_pd else 118.0,
            'hrv_sdnn': 28.0 if is_pd else 50.0,
            'cognitive_load': 0.6 if is_pd else 0.25,
            'source': 'uci'
        })
        uci_ok += 1
    console.print(f'[green]UCI: {uci_ok} records[/]')

# ── Synthetic PD augmentation (if PD samples < 300)
pd_count = sum(r['parkinsons'] for r in records)
if pd_count < 300:
    console.print(f'[yellow]PD samples: {pd_count} → generating synthetic tremor augmentations...[/]')
    synth_dir = DATA / 'synth_pd'; synth_dir.mkdir(exist_ok=True)
    srcs = [r['path'] for r in records if r['source'] in ['ravdess', 'cremad']][:600]
    n_synth = 0
    for i, src in enumerate(srcs):
        try:
            y, sr = librosa.load(src, sr=C.sample_rate, mono=True)
            tremor_freq = np.random.uniform(4.0, 6.5)
            t_arr = np.arange(len(y)) / sr
            tremor = 1.0 + 0.18 * np.sin(2*np.pi*tremor_freq*t_arr)
            y_pd = (y * tremor).astype(np.float32)
            y_pd = librosa.effects.pitch_shift(y_pd, sr=sr, n_steps=np.random.uniform(-0.4, 0.4))
            out_path = synth_dir / f'pd_{i:04d}.wav'
            sf.write(out_path, y_pd, sr)
            sev = np.random.uniform(0.3, 1.0)
            records.append({
                'path': str(out_path), 'parkinsons': 1,
                'depression': 0, 'respiratory': 0,
                'emotion_valence': 0.0, 'emotion_arousal': 0.0,
                'bp_systolic': 132.0 + sev*15, 'hrv_sdnn': 28.0 - sev*10,
                'cognitive_load': 0.5 + sev*0.4, 'source': 'synth_pd'
            })
            n_synth += 1
        except Exception:
            continue
    console.print(f'[green]Generated {n_synth} synthetic PD samples[/]')

df = pd.DataFrame(records)
console.print(f'[bold]Total: {len(df)} | PD: {df.parkinsons.sum()} | Depression: {df.depression.sum()}[/]')
assert len(df) > 0, "No records found — check dataset downloads above"

In [ ]:
smile_extractor = opensmile.Smile(
    feature_set=opensmile.FeatureSet.ComParE_2016,
    feature_level=opensmile.FeatureLevel.Functionals,
)

def extract_smile(path):
    try: return smile_extractor.process_file(path).values[0].astype(np.float32)
    except: return np.zeros(C.smile_dim, dtype=np.float32)

SMILE_CACHE = Path(C.data_dir) / 'smile_cache.npy'
SMILE_IDX   = Path(C.data_dir) / 'smile_idx.json'

if SMILE_CACHE.exists() and SMILE_IDX.exists():
    console.print('[yellow]Loading cached OpenSMILE features...[/]')
    all_smile = np.load(SMILE_CACHE)
    cached_paths = json.load(open(SMILE_IDX))
    path_to_idx = {p: i for i, p in enumerate(cached_paths)}
    df['smile_idx'] = df['path'].map(path_to_idx).fillna(0).astype(int)
    console.print(f'[green]✅ Loaded cache: {all_smile.shape}[/]')
else:
    console.print(f'[cyan]Extracting OpenSMILE for {len(df)} files (~{len(df)//60+1} min)...[/]')
    all_feats = []
    for i, p in enumerate(df['path'].tolist()):
        if i % 100 == 0: console.print(f'  {i}/{len(df)}...')
        all_feats.append(extract_smile(p))
    all_smile = np.stack(all_feats)
    np.save(SMILE_CACHE, all_smile)
    json.dump(df['path'].tolist(), open(SMILE_IDX, 'w'))
    df['smile_idx'] = range(len(df))
    console.print(f'[green]✅ Cache saved: {all_smile.shape}[/]')

scaler_smile = StandardScaler()
train_idx, val_idx = train_test_split(
    np.arange(len(df)), test_size=0.15, random_state=SEED,
    stratify=(df['parkinsons'].astype(str) + df['depression'].astype(str))
)
scaler_smile.fit(all_smile[df.iloc[train_idx]['smile_idx'].values])
all_smile_norm = scaler_smile.transform(all_smile).astype(np.float32)
np.save(Path(C.out_dir)/'smile_mean.npy', scaler_smile.mean_)
np.save(Path(C.out_dir)/'smile_std.npy',  scaler_smile.scale_)
console.print(f'[green]✅ Smile normalised. Train={len(train_idx)} Val={len(val_idx)}[/]')


In [ ]:
def load_audio(path, sr=16000, max_dur=8.0):
    """Load audio. RMS-normalise only — noisereduce removed (too slow for training)."""
    max_s = int(max_dur * sr)
    try:
        y, orig_sr = librosa.load(path, sr=None, mono=True)
        if orig_sr != sr:
            y = librosa.resample(y, orig_sr=orig_sr, target_sr=sr)
        # Fast RMS normalise (replaces slow noisereduce)
        rms = np.sqrt(np.mean(y**2))
        if rms > 1e-6: y = y / (rms + 1e-8) * 0.1
        y = y[:max_s] if len(y) > max_s else np.pad(y, (0, max_s - len(y)))
        return y.astype(np.float32)
    except Exception:
        return np.zeros(max_s, dtype=np.float32)

# Lighter augmentation — PitchShift and TimeStretch are slow, lower probabilities
augment = Compose([
    AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.015, p=0.5),
    TimeStretch(min_rate=0.9, max_rate=1.1, p=0.2),    # was p=0.4
    PitchShift(min_semitones=-1, max_semitones=1, p=0.2),  # was p=0.4
    Shift(min_shift=-0.2, max_shift=0.2, p=0.3),
])

def cutmix_audio(y):
    max_s = len(y)
    cut_len = int(np.random.uniform(0.05, 0.25) * max_s)
    start   = np.random.randint(0, max_s - cut_len)
    y = y.copy(); y[start:start+cut_len] = 0.0
    return y

class VoiceDataset(Dataset):
    def __init__(self, df_split, smile_feats, is_train=True):
        self.df       = df_split.reset_index(drop=True)
        self.smile    = smile_feats
        self.is_train = is_train

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        y = load_audio(row['path'], C.sample_rate, C.max_duration)
        if self.is_train:
            y = augment(samples=y, sample_rate=C.sample_rate)
            if random.random() < 0.3:
                y = np.roll(y, random.randint(-800, 800))
            if random.random() < 0.25:
                y = cutmix_audio(y)
        sidx = int(row.get('smile_idx', i))
        labels = {
            'parkinsons':      torch.tensor(float(row['parkinsons']),          dtype=torch.float32),
            'depression':      torch.tensor(float(row['depression']),          dtype=torch.float32),
            'respiratory':     torch.tensor(float(row['respiratory']),         dtype=torch.float32),
            'emotion_valence': torch.tensor(float(row['emotion_valence']),     dtype=torch.float32),
            'emotion_arousal': torch.tensor(float(row['emotion_arousal']),     dtype=torch.float32),
            'bp_systolic':     torch.tensor(float(row['bp_systolic']) / 200.,  dtype=torch.float32),
            'hrv_sdnn':        torch.tensor(float(row['hrv_sdnn']) / 100.,     dtype=torch.float32),
            'cognitive_load':  torch.tensor(float(row['cognitive_load']),      dtype=torch.float32),
        }
        return {
            'audio':  torch.from_numpy(y),
            'smile':  torch.from_numpy(self.smile[sidx]),
            'labels': labels,
        }

train_df = df.iloc[train_idx]; val_df = df.iloc[val_idx]
train_ds = VoiceDataset(train_df, all_smile_norm, is_train=True)
val_ds   = VoiceDataset(val_df,   all_smile_norm, is_train=False)

pd_labels = train_df['parkinsons'].values.astype(int)
cls_w     = 1.0 / (np.bincount(pd_labels) + 1e-8)
sampler   = WeightedRandomSampler(cls_w[pd_labels], len(pd_labels))

# num_workers=4: avoids all worker crash / shared-memory issues on Kaggle dual T4
# pin_memory=False: conflicts with DataParallel
train_dl = DataLoader(train_ds, batch_size=C.batch_size, sampler=sampler,
                      num_workers=4, pin_memory=False)
val_dl   = DataLoader(val_ds,   batch_size=C.batch_size, shuffle=False,
                      num_workers=4, pin_memory=False)
console.print(f'[bold]Train: {len(train_ds)} | Val: {len(val_ds)}[/]')


In [ ]:
class SmileEncoder(nn.Module):
    def __init__(self, in_dim=6373, h=512, out=256, dp=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, h), nn.GELU(), nn.Dropout(dp),
            nn.Linear(h, h),      nn.GELU(), nn.Dropout(dp),
            nn.Linear(h, out),
        )
        self.norm = nn.LayerNorm(out)
    def forward(self, x): return self.norm(self.net(x))


class Wav2VecLoRAEncoder(nn.Module):
    def __init__(self, model_name='facebook/wav2vec2-large', freeze_layers=20, out_dim=256):
        super().__init__()
        console.print('[cyan]⏳ Loading Wav2Vec2-Large (BF16)...[/]')
        self.w2v = Wav2Vec2Model.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
        )
        console.print('[green]✅ Wav2Vec2-Large loaded[/]')

        console.print(f'[cyan]⏳ Applying LoRA r={C.lora_r}...[/]')
        lora_cfg = LoraConfig(
            r=C.lora_r, lora_alpha=C.lora_alpha,
            lora_dropout=C.lora_dropout,
            target_modules=C.lora_target_modules,
            bias='none',
        )
        self.w2v = get_peft_model(self.w2v, lora_cfg)
        console.print('[green]✅ LoRA applied[/]')

        for p in self.w2v.base_model.feature_extractor.parameters():
            p.requires_grad_(False)
        frozen = 0
        for i, layer in enumerate(self.w2v.base_model.encoder.layers):
            if i < freeze_layers:
                for p in layer.parameters():
                    p.requires_grad_(False)
                frozen += 1
        console.print(f'[green]✅ Frozen {frozen} transformer layers[/]')

        # No gradient_checkpointing — deadlocks on T4 BF16
        self.proj = nn.Sequential(
            nn.Linear(1024, 256), nn.GELU(), nn.LayerNorm(256)
        )

    def forward(self, audio):
        out  = self.w2v(audio, output_hidden_states=False)
        feat = out.last_hidden_state.mean(dim=1)
        return self.proj(feat)


class VoiceBiomarkerV2(nn.Module):
    def __init__(self, cfg: VoiceConfig):
        super().__init__()
        self.cfg       = cfg
        self.w2v_enc   = Wav2VecLoRAEncoder(cfg.w2v_model, cfg.w2v_freeze_layers, 256)
        self.smile_enc = SmileEncoder(cfg.smile_dim, 512, 256, cfg.dropout)
        self.fuse = nn.Sequential(
            nn.Linear(512, cfg.hidden_dim), nn.GELU(), nn.Dropout(cfg.dropout),
            nn.Linear(cfg.hidden_dim, cfg.embed_dim),
        )
        self.norm     = nn.LayerNorm(cfg.embed_dim)
        self.log_vars = nn.Parameter(torch.zeros(len(cfg.all_tasks)))
        self.heads    = nn.ModuleDict()
        for t in cfg.binary_tasks:
            self.heads[t] = nn.Sequential(
                nn.Linear(cfg.embed_dim, 64), nn.GELU(),
                nn.Dropout(cfg.dropout), nn.Linear(64, 1)
            )
        for t in cfg.regress_tasks:
            self.heads[t] = nn.Sequential(
                nn.Linear(cfg.embed_dim, 64), nn.GELU(),
                nn.Dropout(cfg.dropout), nn.Linear(64, 2)
            )

    def forward(self, audio, smile):
        w2v_f = self.w2v_enc(audio)
        sml_f = self.smile_enc(smile)
        embed = self.norm(self.fuse(torch.cat([w2v_f, sml_f], -1)))
        preds = {t: self.heads[t](embed) for t in self.heads}
        return embed, preds, self.log_vars


console.print('[cyan]⏳ Building VoiceBiomarkerV2...[/]')
model = VoiceBiomarkerV2(C).to(DEVICE)

# Enable DataParallel for dual T4
# Safe because: gradient_checkpointing=OFF, GradScaler=disabled
if NUM_GPUS > 1:
    model = nn.DataParallel(model, device_ids=list(range(NUM_GPUS)))
    console.print(f'[bold green]✅ DataParallel across {NUM_GPUS} GPUs[/]')

# No torch.compile — deadlocks with LoRA+BF16 on T4
console.print('[green]✅ Model on device[/]')

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
console.print(f'[bold]Params: {total/1e6:.1f}M total | {trainable/1e6:.2f}M trainable (LoRA)[/]')
console.print('[bold green]✅ Model ready![/]')


In [ ]:
def gaussian_nll_loss(pred_out, target):
    """pred_out: (B,2) — [mean, log_var]; target: (B,)"""
    mean    = pred_out[:, 0]
    log_var = pred_out[:, 1]
    var     = torch.exp(log_var).clamp(min=1e-6)
    return (0.5 * (log_var + (mean - target)**2 / var)).mean()

def kendall_gal_loss(task_loss, log_var):
    """Kendall & Gal 2017 multi-task weighting by learned uncertainty."""
    return torch.exp(-log_var) * task_loss + log_var

bce = nn.BCEWithLogitsLoss()

def compute_loss(preds, labels, log_vars):
    losses = []
    for i, t in enumerate(C.binary_tasks):
        l = bce(preds[t].squeeze(-1), labels[t])
        losses.append(kendall_gal_loss(l, log_vars[i]))
    for j, t in enumerate(C.regress_tasks):
        l = gaussian_nll_loss(preds[t], labels[t])
        losses.append(kendall_gal_loss(l, log_vars[len(C.binary_tasks) + j]))
    return sum(losses)

console.print('[green]✅ Loss functions ready[/]')


In [ ]:
class EMA:
    def __init__(self, m, d=0.9995):
        self.d = d
        raw = m.module if isinstance(m, nn.DataParallel) else m
        self.shadow = {n: p.data.clone().cpu()
                       for n, p in raw.named_parameters() if p.requires_grad}
    def update(self, m):
        raw = m.module if isinstance(m, nn.DataParallel) else m
        for n, p in raw.named_parameters():
            if p.requires_grad and n in self.shadow:
                self.shadow[n] = self.d * self.shadow[n] + (1-self.d) * p.data.cpu()

# ── Unwrap for optimizer param groups / EMA / checkpoint saving
raw_model = model.module if isinstance(model, nn.DataParallel) else model

# ── Optimizer
backbone_params = list(raw_model.w2v_enc.parameters())
other_params    = [p for n, p in raw_model.named_parameters()
                   if not n.startswith('w2v_enc') and p.requires_grad]

try:
    optimiser = bnb.optim.AdamW8bit([
        {'params': backbone_params, 'lr': C.w2v_lr},
        {'params': other_params,    'lr': C.lr},
    ], weight_decay=C.weight_decay)
    console.print('[green]✅ AdamW-8bit (bitsandbytes)[/]')
except Exception as e:
    console.print(f'[yellow]AdamW-8bit failed ({e}), using AdamW FP32[/]')
    optimiser = torch.optim.AdamW([
        {'params': backbone_params, 'lr': C.w2v_lr},
        {'params': other_params,    'lr': C.lr},
    ], weight_decay=C.weight_decay)

total_steps = C.epochs * len(train_dl) // C.grad_accum
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimiser, max_lr=[C.w2v_lr, C.lr],
    total_steps=total_steps, pct_start=0.1,
)
scaler_amp = torch.amp.GradScaler("cuda", enabled=False)
ema = EMA(model, C.ema_decay)

# ── Resume from checkpoint
start_epoch = 1; best_val = float('inf'); history = []
if C.resume_ckpt:
    console.print(f'[cyan]Resuming from {C.resume_ckpt}...[/]')
    try:
        epoch_r, fold_r, ema_shadow, history, best_val = load_checkpoint(
            Path(C.resume_ckpt), raw_model, optimiser, scheduler, scaler_amp
        )
        ema.shadow = ema_shadow
        start_epoch = epoch_r + 1
        console.print(f'[green]✅ Resumed from epoch {epoch_r} (best_val={best_val:.4f})[/]')
    except Exception as e:
        console.print(f'[red]Resume failed: {e} — starting fresh[/]')
        start_epoch = 1

patience_cnt = 0

def move(batch):
    return {
        'audio':  batch['audio'].to(DEVICE),
        'smile':  batch['smile'].to(DEVICE),
        'labels': {k: v.to(DEVICE) for k, v in batch['labels'].items()}
    }

def run_epoch(loader, train=True):
    model.train(train)
    total_loss = 0.0
    all_preds  = {t: [] for t in C.binary_tasks}
    all_labels = {t: [] for t in C.binary_tasks}
    accum_step = 0
    n_batches  = len(loader)
    if train: optimiser.zero_grad()

    for i, batch in enumerate(loader):
        b = move(batch)
        with torch.autocast(device_type='cuda', dtype=DTYPE):
            embed, preds, log_vars = model(b['audio'], b['smile'])
            loss = compute_loss(preds, b['labels'], log_vars) / C.grad_accum

        if train:
            scaler_amp.scale(loss).backward()
            accum_step += 1
            if accum_step % C.grad_accum == 0:
                scaler_amp.unscale_(optimiser)
                nn.utils.clip_grad_norm_(raw_model.parameters(), C.grad_clip)
                scaler_amp.step(optimiser); scaler_amp.update()
                optimiser.zero_grad(); scheduler.step()
                ema.update(model)

        total_loss += loss.item() * C.grad_accum
        for t in C.binary_tasks:
            p = preds[t].squeeze(-1).detach().float().cpu()
            all_preds[t].extend(torch.sigmoid(p).tolist())
            all_labels[t].extend(b['labels'][t].cpu().tolist())

        if i % 50 == 0:
            console.print(f'  [{"train" if train else "val"}] {i}/{n_batches} '
                          f'loss={loss.item()*C.grad_accum:.4f}')

    aucs = {}
    for t in C.binary_tasks:
        y, p = np.array(all_labels[t]), np.array(all_preds[t])
        aucs[t] = roc_auc_score(y, p) if len(np.unique(y)) > 1 else 0.5
    return total_loss / len(loader), aucs


console.print(f'[bold]Training {C.epochs} epochs | start={start_epoch} | grad_accum={C.grad_accum} | GPUs={NUM_GPUS}[/]')

for epoch in range(start_epoch, C.epochs + 1):
    t0 = time.time()
    console.print(f'\n[bold cyan]── Epoch {epoch}/{C.epochs} ──[/]')

    tr_loss, tr_aucs = run_epoch(train_dl, train=True)
    with torch.no_grad():
        vl_loss, vl_aucs = run_epoch(val_dl, train=False)

    history.append({'epoch': epoch, 'train_loss': tr_loss, 'val_loss': vl_loss,
                    **{f'val_{t}': v for t, v in vl_aucs.items()}})
    mean_auc = np.mean(list(vl_aucs.values()))
    console.print(
        f'Ep {epoch:02d}/{C.epochs} | tr={tr_loss:.4f} vl={vl_loss:.4f} | '
        f'PD={vl_aucs["parkinsons"]:.3f} DEP={vl_aucs["depression"]:.3f} | '
        f'meanAUC={mean_auc:.3f} | {time.time()-t0:.0f}s'
    )

    save_checkpoint(
        CKPT_DIR, epoch, raw_model.state_dict(),
        ema.shadow, optimiser.state_dict(), scheduler.state_dict(),
        scaler_amp.state_dict(), history, vl_loss
    )

    if vl_loss < best_val:
        best_val = vl_loss; patience_cnt = 0
        torch.save({
            'model_state': raw_model.state_dict(),
            'ema_shadow': ema.shadow, 'val_aucs': vl_aucs, 'epoch': epoch
        }, CKPT_DIR / 'voice_best.pt')
        console.print('  [green]✅ Best model saved[/]')
    else:
        patience_cnt += 1
        if patience_cnt >= C.patience:
            console.print(f'[yellow]Early stop @ epoch {epoch}[/]'); break

json.dump(history, open(Path(C.out_dir)/'voice_history.json','w'), indent=2)
console.print('[bold green]\n✅ Training complete![/]')

In [ ]:
def expected_calibration_error(y_true, y_prob, n_bins=10):
    bins = np.linspace(0, 1, n_bins+1)
    ece  = 0.0; n = len(y_true)
    for b in range(n_bins):
        mask = (y_prob >= bins[b]) & (y_prob < bins[b+1])
        if mask.sum() == 0: continue
        ece += mask.sum() / n * abs(y_true[mask].mean() - y_prob[mask].mean())
    return ece

# Load best model — unwrap DataParallel before loading state dict
raw_model_eval = model.module if isinstance(model, nn.DataParallel) else model
ckpt = torch.load(CKPT_DIR / 'voice_best.pt', map_location=DEVICE, weights_only=False)
raw_model_eval.load_state_dict(ckpt['model_state'])
# Apply EMA weights
for n, p in raw_model_eval.named_parameters():
    if p.requires_grad and n in ckpt['ema_shadow']:
        p.data.copy_(ckpt['ema_shadow'][n].to(DEVICE))
model.eval()

all_p = {t: [] for t in C.binary_tasks}
all_y = {t: [] for t in C.binary_tasks}
all_embeds = []

with torch.no_grad():
    for batch in val_dl:
        b = move(batch)
        with torch.autocast(device_type='cuda', dtype=DTYPE):
            embed, preds, _ = model(b['audio'], b['smile'])
        all_embeds.append(embed.float().cpu())
        for t in C.binary_tasks:
            all_p[t].extend(torch.sigmoid(preds[t].squeeze(-1)).float().cpu().tolist())
            all_y[t].extend(b['labels'][t].cpu().tolist())

all_embeds = torch.cat(all_embeds, 0).numpy()

fig, axes = plt.subplots(1, len(C.binary_tasks), figsize=(5*len(C.binary_tasks), 5))
if len(C.binary_tasks) == 1: axes = [axes]

for ax, t in zip(axes, C.binary_tasks):
    y_true = np.array(all_y[t]); y_prob = np.array(all_p[t])
    if len(np.unique(y_true)) < 2: continue
    frac_pos, mean_pred = calibration_curve(y_true, y_prob, n_bins=10)
    ece = expected_calibration_error(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    ax.plot(mean_pred, frac_pos, 's-', label=f'Model (ECE={ece:.3f})')
    ax.plot([0,1],[0,1],'k--', label='Perfect')
    ax.set_title(f'{t}\nAUC={auc:.3f}')
    ax.set_xlabel('Mean predicted prob'); ax.set_ylabel('Fraction positives')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle('BioEcho v2 — Reliability Diagrams (ECE Calibration)', fontsize=13)
plt.tight_layout()
plt.savefig(Path(C.out_dir)/'calibration.png', dpi=150)
plt.show()

tbl = Table(title='Voice v2 — Eval Metrics', show_lines=True)
for col in ['Task','AUC-ROC','AP','F1@0.5','ECE']: tbl.add_column(col)
for t in C.binary_tasks:
    y, p = np.array(all_y[t]), np.array(all_p[t])
    if len(np.unique(y)) < 2: continue
    tbl.add_row(t,
        f'{roc_auc_score(y,p):.3f}',
        f'{average_precision_score(y,p):.3f}',
        f'{f1_score(y,(p>0.5).astype(int)):.3f}',
        f'{expected_calibration_error(y,p):.3f}')
console.print(tbl)


In [ ]:
class VoiceExportWrapper(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, audio, smile):
        embed, preds, _ = self.m(audio, smile)
        bin_stack = torch.stack([preds[t].squeeze(-1) for t in C.binary_tasks], -1)
        reg_stack = torch.stack([preds[t][:, 0]       for t in C.regress_tasks], -1)
        return embed, bin_stack, reg_stack

# Unwrap DataParallel before export
raw_for_export = model.module if isinstance(model, nn.DataParallel) else model
raw_m_cpu = deepcopy(raw_for_export).float().cpu().eval()
export_m  = VoiceExportWrapper(raw_m_cpu).eval()

dummy_audio = torch.randn(1, int(C.max_duration * C.sample_rate))
dummy_smile = torch.randn(1, C.smile_dim)

fp32_path = Path(C.out_dir) / 'voice_fp32.onnx'
console.print('[cyan]⏳ Exporting FP32 ONNX...[/]')
torch.onnx.export(
    export_m, (dummy_audio, dummy_smile), fp32_path,
    input_names=['audio', 'smile_features'],
    output_names=['voice_embedding', 'binary_risks', 'regression_scores'],
    dynamic_axes={k: {0: 'batch'} for k in [
        'audio','smile_features','voice_embedding','binary_risks','regression_scores'
    ]},
    opset_version=17, do_constant_folding=True
)
onnx.checker.check_model(onnx.load(str(fp32_path)))
console.print(f'[green]✅ FP32 ONNX: {fp32_path.stat().st_size/1e6:.1f} MB[/]')

int8_path = Path(C.out_dir) / 'voice_int8.onnx'
quantize_dynamic(str(fp32_path), str(int8_path), weight_type=QuantType.QInt8, optimize_model=True)
console.print(f'[green]✅ INT8 ONNX: {int8_path.stat().st_size/1e6:.1f} MB ' +
              f'({fp32_path.stat().st_size/int8_path.stat().st_size:.1f}× smaller)[/]')

inp = {'audio': dummy_audio.numpy(), 'smile_features': dummy_smile.numpy()}

sess_gpu = ort.InferenceSession(str(fp32_path), providers=['CUDAExecutionProvider','CPUExecutionProvider'])
for _ in range(5): sess_gpu.run(None, inp)
t0 = time.time(); [sess_gpu.run(None, inp) for _ in range(50)]
ms_gpu = (time.time()-t0)/50*1000

sess_cpu = ort.InferenceSession(str(int8_path), providers=['CPUExecutionProvider'])
for _ in range(5): sess_cpu.run(None, inp)
t0 = time.time(); [sess_cpu.run(None, inp) for _ in range(20)]
ms_cpu = (time.time()-t0)/20*1000

tbl = Table(title='Inference Benchmark', show_lines=True)
for c in ['Backend','ms/call','FPS']: tbl.add_column(c)
tbl.add_row('T4 GPU (FP32)', f'{ms_gpu:.1f}', f'{1000/ms_gpu:.0f}')
tbl.add_row('CPU INT8',      f'{ms_cpu:.1f}', f'{1000/ms_cpu:.0f}')
console.print(tbl)

console.print('[bold green]\n✅ Voice notebook complete![/]')
console.print(f'  Embedding dim : {C.embed_dim}')
console.print(f'  Binary tasks  : {C.binary_tasks}')
console.print(f'  Regress tasks : {C.regress_tasks}')
console.print(f'  INT8 ONNX     : {int8_path}')
